# 02 · Make one CSV per sequencing facility (healthy controls appended)

**Run order for the RNA-seq workflow:** **02** (this notebook) → **03** select samples + patient metadata → **04** batch PCA + ComBat-seq.
Kernel: **Python 3**.

**What this does.** The Goldbach-Mansky workbook has one tab per sequencing facility (BCM-HGSC, NCI-FNL, NIAID-NCI, NIAMS).
This notebook writes each tab to its own CSV, then adds each healthy control (from `HC_blood_all_samples.xlsx`) to the CSV of the
facility it was sequenced at (its own `Sequencing_Facility` line). Genes are lined up by `Gene_ID`; if the gene lists differ it stops
instead of filling zeros. Blank count cells are kept blank (never turned into 0) and reported; notebook 03 excludes those samples.

**Reads:** the two workbooks in `/mnt/file-systems/`. **Writes:** `rnaseq_out_v2/facility_csv/` (four CSVs + `facility_csv_summary.csv`).
It is skipped on re-run unless a workbook is newer than the CSVs (set `FORCE = True` to rebuild).

In [ ]:
import os, re
import numpy as np
import pandas as pd

MNT = "/mnt/file-systems/"
GM_XLSX = MNT + "Goldbach-Mansky_compiled_RNA-Seq_GeneCounts_CANDLE_AND_SAVI_SUBSET_2026_09_22.xlsx"
HC_XLSX = MNT + "HC_blood_all_samples.xlsx"
RESULTS = "/home/jovyan/session_data/Attempt0/endotype_results"
OUT_DIR = RESULTS + "/rnaseq_out_v2/facility_csv"
FORCE = False

INFO_LABELS = ["Enrollment ID", "Metadata Initials", "Sample Source", "Diagnosis", "Treatment",
               "Sequencing_Facility", "Sequencing_Cohort", "Gender", "RNA sample collection date"]
N_INFO = len(INFO_LABELS)
os.makedirs(OUT_DIR, exist_ok=True)

### Helpers
Each sheet has the layout: line 1 = `Gene_ID, Gene_Name, Stat, <samples>`; the next 9 lines are sample information; then one gene per line (`Raw_Count`).
`read_tab` checks that layout and the counts (whole numbers, 0 or more). `facility_key` treats the spellings `BMC-HGSC` and `BCM-HGSC` as the same facility.

In [ ]:
def facility_key(name):
    k = re.sub(r"[^A-Z0-9]", "", str(name).upper())
    return k.replace("BMCHGSC", "BCMHGSC")

def read_tab(df):
    hdr = [str(v).strip() for v in df.iloc[0, :3]]
    assert hdr == ["Gene_ID", "Gene_Name", "Stat"], f"unexpected header {hdr}"
    samples = [str(v) for v in df.iloc[0, 3:]]
    assert len(set(samples)) == len(samples), "duplicate sample column names inside one sheet"
    labels = [str(v).strip() for v in df.iloc[1:1 + N_INFO, 2]]
    assert labels == INFO_LABELS, f"sample-info lines are {labels}, expected {INFO_LABELS}"
    info = df.iloc[1:1 + N_INFO, 3:].copy(); info.index = INFO_LABELS; info.columns = samples
    body = df.iloc[1 + N_INFO:, :].copy()
    assert (body.iloc[:, 2].astype(str).str.strip() == "Raw_Count").all(), "a gene line is not Raw_Count"
    genes = body.iloc[:, :2].copy(); genes.columns = ["Gene_ID", "Gene_Name"]; genes = genes.reset_index(drop=True)
    assert not genes["Gene_ID"].duplicated().any(), "Gene_ID repeated inside one sheet"
    counts = body.iloc[:, 3:].apply(pd.to_numeric, errors="raise"); counts.columns = samples
    counts.index = genes["Gene_ID"].values
    v = counts.to_numpy(dtype=float); v = v[~np.isnan(v)]          # blank cells are allowed (reported below, handled in 03)
    assert (v >= 0).all(), "negative counts"
    assert (np.mod(v, 1) == 0).all(), "non-integer counts"
    return info, genes, counts.astype("Int64")

def read_book(path):
    out = {}
    for n, d in pd.read_excel(path, sheet_name=None, header=None, dtype=object).items():
        info, genes, counts = read_tab(d)
        miss = counts.isna()
        print(f"{os.path.basename(path)[:30]} [{n}]: {counts.shape[1]} samples, {len(genes)} genes; "
              f"samples with no counts at all: {int(miss.all().sum())}; "
              f"samples with some blank counts: {int((miss.any() & ~miss.all()).sum())}")
        out[n] = (info, genes, counts)
    return out

def write_csv(path, info, genes, counts):
    samples = list(counts.columns)
    head = pd.DataFrame([["", "Gene_ID", "Gene_Name", "Stat"] + samples])
    top = pd.DataFrame([[i + 1, "-", "-", lab] + list(info.loc[lab, samples]) for i, lab in enumerate(INFO_LABELS)])
    body = pd.DataFrame({0: range(N_INFO + 1, N_INFO + 1 + len(genes)), 1: genes["Gene_ID"].values,
                         2: genes["Gene_Name"].values, 3: "Raw_Count"})
    body = pd.concat([body, counts.reset_index(drop=True)], axis=1, ignore_index=True)
    top.columns = head.columns = body.columns = range(body.shape[1])
    pd.concat([head, top, body], ignore_index=True).to_csv(path, index=False, header=False)

### Build the facility CSVs
Prints one table: samples per facility tab, healthy controls appended, total, genes. A healthy control whose facility matches no tab is **not** appended and is reported.

In [ ]:
summary_csv = os.path.join(OUT_DIR, "facility_csv_summary.csv")
newest_input = max(os.path.getmtime(GM_XLSX), os.path.getmtime(HC_XLSX))
up_to_date = os.path.exists(summary_csv) and os.path.getmtime(summary_csv) > newest_input

if up_to_date and not FORCE:
    print("facility CSVs are newer than the workbooks: nothing to do (FORCE = True to rebuild)")
else:
    gm, hc_tabs = read_book(GM_XLSX), read_book(HC_XLSX)

    hc_info, hc_counts, hc_genes = [], [], None
    for name, (info, genes, counts) in hc_tabs.items():            # pool every HC tab
        if hc_genes is None: hc_genes = genes
        else: assert set(genes["Gene_ID"]) == set(hc_genes["Gene_ID"]), f"HC tab '{name}' lists different genes"
        hc_info.append(info); hc_counts.append(counts.loc[hc_genes["Gene_ID"]])
    hc_info, hc_counts = pd.concat(hc_info, axis=1), pd.concat(hc_counts, axis=1)
    assert not hc_counts.columns.duplicated().any(), "same sample column name in two HC tabs"

    tab_key = {facility_key(n): n for n in gm}
    assert len(tab_key) == len(gm), "two tab names look like the same facility"
    hc_tab = hc_info.loc["Sequencing_Facility"].map(lambda v: tab_key.get(facility_key(v)))
    unplaced = hc_tab[hc_tab.isna()].index.tolist()

    rows = []
    for tab, (info, genes, counts) in gm.items():
        want = list(hc_tab[hc_tab == tab].index)
        mine = [c for c in want if c not in counts.columns]
        assert set(genes["Gene_ID"]) == set(hc_genes["Gene_ID"]), f"{tab}: HC workbook and facility tab list different genes"
        info2 = pd.concat([info, hc_info[mine]], axis=1)
        counts2 = pd.concat([counts, hc_counts.loc[genes["Gene_ID"], mine]], axis=1)
        out = os.path.join(OUT_DIR, re.sub(r"[^A-Za-z0-9_.-]", "_", tab) + ".csv")
        write_csv(out, info2, genes, counts2)
        rows.append({"facility_tab": tab, "file": os.path.basename(out), "samples_in_tab": counts.shape[1],
                     "HC_appended": len(mine), "HC_already_in_tab_skipped": len(want) - len(mine),
                     "samples_total": counts2.shape[1], "genes": len(genes)})
    summary = pd.DataFrame(rows)
    summary.to_csv(summary_csv, index=False)
    print(summary.to_string(index=False))
    if unplaced:
        print(f"\nWARNING: {len(unplaced)} HC samples have a Sequencing_Facility that matches no tab "
              f"({sorted(set(hc_info.loc['Sequencing_Facility', unplaced].astype(str)))}) and were NOT appended.")
    else:
        print(f"\nAll {hc_counts.shape[1]} HC samples placed.")

**Next:** open **03_NIH_bulk_RNA_seq_parse_metadata.ipynb**.